In [0]:
from pyspark.sql import functions as F

cat = "capstone6_telecom.telecom"

def n(sql):
    return spark.sql(sql).collect()[0][0]

checks = []
def check(test, metric, actual, expected):
    checks.append((test, metric, int(actual), int(expected), "PASS" if int(actual) == int(expected) else "FAIL"))

In [0]:
b_ev  = n(f"SELECT COUNT(*) FROM {cat}.bronze_network_events")
s_ev  = n(f"SELECT COUNT(*) FROM {cat}.silver_network_events")
r_ev  = n(f"SELECT COUNT(*) FROM {cat}.silver_rejected_records WHERE record_type='EVENT'")
check("TC01", "bronze = silver + rejected/deduped (events)", b_ev, s_ev + r_ev)
check("TC01", "silver events", s_ev, 11500)
check("TC01", "duplicates removed", n(f"SELECT COUNT(*) FROM {cat}.silver_rejected_records WHERE record_type='EVENT' AND rule_id='DQ08'"), 40)
check("TC01", "duplicate ids left in silver", n(f"SELECT COUNT(*)-COUNT(DISTINCT measurement_id) FROM {cat}.silver_network_events"), 0)
check("TC02", "network_type set to UNKNOWN", n(f"SELECT COUNT(*) FROM {cat}.silver_network_events WHERE network_type_missing"), 12)
check("TC02", "latency outliers flagged (kept)", n(f"SELECT COUNT(*) FROM {cat}.silver_network_events WHERE latency_outlier_flag"), 35)
check("TC07", "events with unknown region_id", n(f"""SELECT COUNT(*) FROM {cat}.silver_network_events e
    LEFT ANTI JOIN {cat}.silver_dim_region r ON e.region_id = r.region_id"""), 0)
check("TC07", "events with unknown tower_id", n(f"""SELECT COUNT(*) FROM {cat}.silver_network_events e
    LEFT ANTI JOIN {cat}.silver_dim_tower t ON e.tower_id = t.tower_id"""), 0)

b_cdc = n(f"SELECT COUNT(*) FROM {cat}.bronze_status_updates")
h     = n(f"SELECT COUNT(*) FROM {cat}.silver_status_history")
r_cdc = n(f"SELECT COUNT(*) FROM {cat}.silver_rejected_records WHERE record_type='CDC'")
check("TC04", "bronze CDC = history + rejected", b_cdc, h + r_cdc)
check("TC04", "history unique (measurement_id, sequence)", n(f"SELECT COUNT(*) - COUNT(DISTINCT measurement_id, source_sequence) FROM {cat}.silver_status_history"), 0)
check("TC04", "current rows = events", n(f"SELECT COUNT(*) FROM {cat}.silver_network_current"), s_ev)
check("TC04", "current status disagrees with latest history", n(f"""
    SELECT COUNT(*) FROM {cat}.silver_network_current c
    JOIN (SELECT measurement_id, MAX_BY(new_status, source_sequence) AS latest FROM {cat}.silver_status_history GROUP BY 1) h
      USING (measurement_id) WHERE c.current_status <> h.latest"""), 0)

b_sec = n(f"SELECT COUNT(*) FROM {cat}.bronze_secondary_events")
s_sec = n(f"SELECT COUNT(*) FROM {cat}.silver_secondary_events")
r_sec = n(f"SELECT COUNT(*) FROM {cat}.silver_rejected_records WHERE record_type='SECONDARY'")
check("JOIN", "bronze secondary = silver + rejected", b_sec, s_sec + r_sec)
check("JOIN", "enriched rows = events (no fan-out)", n(f"SELECT COUNT(*) FROM {cat}.silver_network_enriched"), s_ev)

check("OUTAGE", "interval failed readings = current FAILED readings",
      n(f"SELECT COALESCE(SUM(failed_readings),0) FROM {cat}.silver_outage_intervals"),
      n(f"SELECT COUNT(*) FROM {cat}.silver_network_enriched WHERE current_status='FAILED'"))

In [0]:
(spark.createDataFrame(checks, "test_id string, metric string, actual long, expected long, result string")
      .withColumn("checked_at", F.current_timestamp())
      .write.format("delta").mode("overwrite").option("overwriteSchema", "true")
      .saveAsTable(f"{cat}.silver_validation_summary"))
display(spark.table(f"{cat}.silver_validation_summary"))